In [ ]:
import math
import random
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers # type: ignore
import numpy as np
import pandas_datareader as pdr
from tqdm import tqdm_notebook, tqdm
from collections import deque

In [ ]:
class IA_Trader:
    def __init__(self, state_size, action_space=3, model_name='AITrader'):
        """
        state_size: São os estados/informações que vêm do ambiente
        action_space: Número de ações possíveis
        """
        self.state_size = state_size
        self.action_space = action_space
        self.memory = deque(maxlen=2000)
        self.model_name = model_name

        # Parâmetros a seguir vêm da fórmula do algoritmo
        self.gamma = 0.95  # Fator de desconto
        self.epsilon = 1  # (100%) ações randômicas sem usar a rede neural
        self.epsilon_decay = 0.995
        self.epsilon_final = 0.01

        self.model = self.model_builder()

    def model_builder(self):
        model = tf.keras.models.Sequential([
            layers.Dense(32, activation="relu", input_dim=self.state_size),
            layers.Dense(64, activation="relu"),
            layers.Dense(128, activation="relu"),
            layers.Dense(units=self.action_space, activation="linear")
        ])
        model.compile(loss="mse", optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

        return model

    def trade(self, state):
        if random.random() <= self.epsilon:
            return random.randrange(self.action_space)

        actions = self.model.predict(state)
        return np.argmax(actions[0])

    def batch_train(self, batch_size):
        batch = []
        for i in range(len(self.memory) - batch_size + 1, len(self.memory)):
          batch.append(self.memory[i])
        for state, action, reward, next_state, done in batch:
            if not done:
                reward = reward + self.gamma * np.amax(self.model.predict(next_state)[0])

            target = self.model.predict(state)
            target[0][action] = reward

            self.model.fit(state, target, epochs=1, verbose=0)

        if self.epsilon > self.epsilon_final:
            self.epsilon *= self.epsilon_decay

In [ ]:
def sigmoid(x):
  return 1 / (1 + math.exp(-x))

In [ ]:
def stocks_price_format(n):
    if n < 0:
        return f"- $ {abs(n):.2f}"
    else:
        return f"$ {abs(n):.2f}"

In [ ]:
dataset = pdr.DataReader(
    "AAPL", data_source="stooq", start="2012-01-01", end="2024-03-01")

In [ ]:
def dataset_loader(stock_name):
  dataset = pdr.DataReader(
    stock_name, data_source="stooq")

  start_date = str(dataset.index[0]).split()[0]
  stop_date = str(dataset.index[-1]).split()[0]
  close = dataset['Close']

  return close

In [ ]:
def state_creator(data, timestep, window_size):

  starting_id = timestep - window_size + 1

  if starting_id >= 0:
    windowed_data = data[starting_id:timestep + 1]
  else:
    windowed_data = - starting_id * [data[0]] + list(data[0:timestep + 1])

  state = []
  for i in range(window_size - 1):
    state.append(sigmoid(windowed_data[i + 1] - windowed_data[i]))

  return np.array([state]), windowed_data

In [ ]:
stock_name = "AAPL"
data =  dataset_loader(stock_name)

In [ ]:
state, window  = state_creator(data, 0, 5)

In [ ]:
window_size = 10
episodes = 1000
batch_size = 32
data_sample = len(data) -1

In [ ]:
trader = IA_Trader(window_size)

In [ ]:
trader.model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_24 (Dense)            (None, 32)                352       
                                                                 
 dense_25 (Dense)            (None, 64)                2112      
                                                                 
 dense_26 (Dense)            (None, 128)               8320      
                                                                 
 dense_27 (Dense)            (None, 3)                 387       
                                                                 
Total params: 11171 (43.64 KB)
Trainable params: 11171 (43.64 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
for episode in range(1, episodes + 1):
    print(f"Episódio {episode}: {episodes}")
    state, _ = state_creator(data, 0, window_size + 1)
    total_profit = 0
    trader.inventory = []

    for t in tqdm(range(data_sample)):
        action = trader.trade(state)
        next_state, _ = state_creator(data, t + 1, window_size + 1)
        reward = 0

        if action == 1:  # Comprando ação
            trader.inventory.append(data[t])
            print("AI Trader bought:", stocks_price_format(data[t]))

        elif action == 2 and len(trader.inventory) > 0:  # Vendendo uma ação
            buy_price = trader.inventory.pop(0)
            reward = max(data[t] - buy_price, 0)
            total_profit += data[t] - buy_price
            print(f"AI Trader sold", stocks_price_format(data[t]), "Profit:" + stocks_price_format(data[t] - buy_price))

        if t == data_sample - 1:
            done = True
        else:
            done = False

        trader.memory.append((state, action, reward, next_state, done))
        state = next_state

        if done:
            print("############################################################")
            print(f"Total profit: {total_profit}")
            print("############################################################")

        if len(trader.memory) > batch_size:
            trader.batch_train(batch_size)
    if episode % 10 == 0:
      trader.model.save(f"Ai_Trader_Episode_{episode}.h5")